# Aufgabe 0: Wie funktioniert ein LLM-API-Call?

Voraussetzung: Setup aus README abgeschlossen.

In [ ]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b nicht gefunden: {models}"
print("OK")

## Teil 1: Ein LLM-Call

Ein LLM generiert Text, Token fuer Token. Alles was wir in APIs wie `ollama.chat()` sehen (Felder wie `thinking`, `content`, `tool_calls`) ist nachtraeglich aus diesem Text-Stream geparst.

Zuerst schauen wir uns den rohen Output an, direkt ueber die REST API.

In [ ]:
# Roher LLM-Output direkt ueber die REST API
# raw=true: kein Parsing durch Ollama, wir sehen den Text wie er generiert wird
# Prompt im Qwen3.5 Chat-Template mit <think> pre-fill (aktiviert den Think-Modus)
import requests, json

OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "<|im_start|>user\nWas ist 2 + 2?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "raw": True,
    "stream": False,
})

# <think> war pre-filled, also vorne dranhaengen fuer das vollstaendige Bild
raw_output = "<think>\n" + resp.json()["response"]
# print(repr(raw_output))
print(raw_output)

In [ ]:
# Ohne Chat Template: das Modell weiss nicht wo die Antwort aufhoeren soll
# Nach 3 Sekunden brechen wir ab
import time

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "Was ist 2 + 2?",
    "raw": True,
    "stream": True,
}, stream=True)

tokens = []
start = time.time()
for line in resp.iter_lines():
    if time.time() - start > 3:
        break
    tokens.append(json.loads(line)["response"])

# print(repr("".join(tokens)))
print("".join(tokens))

In [ ]:
# ollama.chat() macht das gleiche, aber:
# 1. Baut den Prompt mit Special Tokens automatisch (RENDERER)
# 2. Parst <think>...</think> aus dem Output in das Feld "thinking" (PARSER)
# 3. Der Rest wird zu "content"
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Was ist 2 + 2?"}],
)

print(json.dumps(response.message.model_dump(), indent=2, default=str))

## Teil 2: Statelessness

Die Chat API ist stateless. Das Modell sieht bei jedem Call nur die `messages` die wir in diesem Call mitschicken.

In [ ]:
# Ohne Conversation History: zwei getrennte Calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Ich heisse Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Wie heisse ich?"}])
print("Call 2:", r2.message.content)

In [ ]:
# Mit Conversation History: gleiche zwei Calls, aber History mitschicken
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Ich heisse Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "Ich heisse Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "Wie heisse ich?"},
])
print("Call 2:", r2.message.content)

## Teil 3: Tool Calling

Ein LLM generiert immer nur Text. Tool Calling ist kein spezielles Feature, sondern eine **Konvention fuer strukturierten Output**: das Modell wurde darauf trainiert, bei bestimmten Prompts JSON statt Freitext auszugeben.

Das koennen wir zuerst von Hand nachbauen, nur mit einem System Prompt.

In [ ]:
# "Tool Calling" ohne tools= Parameter, nur mit Prompt
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": 'Wenn du rechnen musst, antworte NUR mit diesem JSON:\n{"tool": "calculator", "expression": "<ausdruck>"}\nKein anderer Text.'},
        {"role": "user", "content": "Berechne 17 * 23"},
    ],
)

print("=== Raw Output (nur Text) ===")
print(response.message.content)

In [ ]:
# Wir koennen den Text-Output selbst parsen und das Tool ausfuehren
import json as _json

parsed = _json.loads(response.message.content)
print("Parsed:", parsed)

# Das ist Tool Calling: das Modell gibt strukturierten Text aus, wir parsen und fuehren aus.
result = eval(parsed["expression"])
print("Result:", result)

Das funktioniert, ist aber fragil: wir muessen den Prompt genau formulieren, das JSON-Format selbst definieren und den Output selbst parsen. Der `tools=` Parameter automatisiert genau das:
- Er injiziert die Tool Definitions in den Prompt (im Format das das Modell beim Training gesehen hat)
- Er parst den Output und liefert strukturierte `tool_calls` statt rohem Text

Ab hier nutzen wir `tools=`. Die Tool Definition beschreibt dem Modell welche Funktionen verfuegbar sind. Die Tool Implementation ist die Python-Funktion die wir ausfuehren.

In [ ]:
# Tool Definition: beschreibt dem Modell welche Funktion verfuegbar ist
# Referenz: https://github.com/ollama/ollama/blob/main/docs/api.md#chat-request-with-tools
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

In [ ]:
# LLM mit Tool Definition aufrufen, volle Response anschauen
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Berechne 17 * 23"}],
    tools=calculator_tool,
)

print(json.dumps(response.model_dump(), indent=2, default=str))

In der Response sehen wir:
- `content` ist leer: das Modell hat keinen Text zurueckgegeben
- `tool_calls` ist gefuellt: das Modell will `calculator` mit `{"expression": "17 * 23"}` aufrufen

Das Modell hat nichts berechnet. Es hat zurueckgegeben **welche Funktion** es mit **welchen Argumenten** aufrufen will. Jetzt brauchen wir die Python-Funktion die das tatsaechlich ausfuehrt.

In [ ]:
# Tool Implementation: die Python-Funktion die wir ausfuehren
import numexpr, math

def calculator(expression: str) -> str:
    result = numexpr.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
    return f"Result: {float(result)}"

# Test: direkt aufrufen
print(calculator("17 * 23"))

In [ ]:
# Verbindung: wie finden wir die richtige Funktion fuer einen Tool Call?
# Der Name im Tool Call ("calculator") muss mit dem Key im tool_map uebereinstimmen.

tool_map = {"calculator": calculator}

tc = response.message.tool_calls[0]       # Tool Call aus der Response
print(f"name:      {tc.function.name}")    # "calculator"
print(f"arguments: {tc.function.arguments}")  # {"expression": "17 * 23"}

func = tool_map[tc.function.name]          # calculator Funktion nachschlagen
result = func(**tc.function.arguments)     # calculator(expression="17 * 23")
print(f"result:    {result}")

## Teil 4: Tool-Use Loop

Wir haben das Tool ausgefuehrt und ein Ergebnis. Aber die API ist stateless: das Modell weiss nichts von der Ausfuehrung. Wir muessen das Ergebnis als `{"role": "tool"}` Message in die History aufnehmen und das Modell nochmal aufrufen.

In [ ]:
# Kompletter Tool-Use Loop
messages = [{"role": "user", "content": "Berechne 17 * 23"}]

# 1. LLM aufrufen
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Pruefen: will das Modell ein Tool aufrufen?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Tool ausfuehren
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    result = func(**tc.function.arguments)

    # 4. Tool-Ergebnis zurueck ans Modell schicken
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages an LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Finale Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

In [ ]:
# Gegenprobe: was passiert wenn das Modell sich GEGEN ein Tool entscheidet?
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Berechne 17 * 23. Antworte direkt, benutze KEIN Tool."}],
    tools=calculator_tool,
)

print("tool_calls:", response.message.tool_calls)
print("content:   ", json.dumps(response.model_dump(), indent=2, default=str))

## Zusammenfassung

1. `ollama.chat()` nimmt eine `messages`-Liste und gibt ein `ChatResponse`-Objekt zurueck.
2. Die API ist stateless. Das Modell sieht nur die Messages die wir in diesem Call mitschicken.
3. Mit `tools=` kann das Modell Tool Calls zurueckgeben. Ob es das tut pruefen wir mit `if response.message.tool_calls`.
4. Die Tool Definition (JSON-Objekt mit Name, Description, Parameter-Schema) beschreibt dem Modell was verfuegbar ist. Die Tool Implementation (Python-Funktion) fuehren wir selbst aus. Verbunden werden sie ueber `tool_map`.
5. Der Tool-Use Loop (LLM aufrufen → pruefen → Tool ausfuehren → Ergebnis zurueck → LLM nochmal aufrufen) ist das was ein Agent automatisiert.

→ **Aufgabe 1**: diesen Loop implementieren.